# P&ID Stage 1 — Train on Colab (GPU)

**Luồng**: code từ GitHub (`git clone`) + data từ Google Drive (mount). Data KHÔNG đi qua git.

**Chuẩn bị 1 lần trước khi chạy notebook:**
1. Push code lên GitHub (xem `docs/PUSH_AND_COLAB.md`).
2. Upload thư mục `data/canonical/` (ảnh + `annotations.coco.json` + `digitizepid_splits.json`) lên Google Drive tại: `MyDrive/pid/data/canonical/`.

**Runtime**: Runtime → Change runtime type → **GPU (T4)**.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive (chứa data + nơi lưu checkpoint)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Sửa nếu bạn đặt data ở path khác trên Drive:
DRIVE_ROOT = '/content/drive/MyDrive/pid'
DATA_ROOT  = f'{DRIVE_ROOT}/data'        # phải chứa canonical/
RUNS_DIR   = f'{DRIVE_ROOT}/runs'         # checkpoint sống qua các session
assert os.path.isdir(f'{DATA_ROOT}/canonical'), f'Không thấy {DATA_ROOT}/canonical — upload data lên Drive trước.'
print('canonical images:', len(os.listdir(f'{DATA_ROOT}/canonical/images')))

## 3. Clone code từ GitHub + cài deps (uv)

In [ ]:
REPO_URL = 'https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git'  # <-- sửa
%cd /content
![ -d pid ] || git clone $REPO_URL pid
%cd /content/pid
!git pull
!pip install -q uv
# Colab đã có torch CUDA; --no-install-project để khỏi build wheel, chỉ cài deps.
!uv pip install --system -q ultralytics sahi ensemble-boxes albumentations timm hydra-core omegaconf pycocotools opencv-python-headless pillow pandas scikit-learn matplotlib tqdm pyyaml

## 4. Cấu hình đường dẫn

`src/pipeline` + `src/utils` là flat layout — thêm `src/` vào PYTHONPATH. Data đọc từ Drive, checkpoint ghi ra Drive.

In [ ]:
import os
os.environ['PYTHONPATH'] = '/content/pid/src'
os.environ['PID_DATA_ROOT'] = DATA_ROOT
os.environ['PID_CONFIG_DIR'] = '/content/pid/configs'
print('PID_DATA_ROOT =', DATA_ROOT)
print('RUNS_DIR      =', RUNS_DIR)

## 5. Data prep — chạy MỘT LẦN (kết quả lưu trên Drive)

`classes` → `manifest` → `yolo` → `tiles`. Bước `tiles` cắt 500 ảnh 7168×4561 thành ~27k tile 1024px (nặng nhất, ~10-20 phút). Vì output nằm trên Drive (`derived/`), các session sau **bỏ qua cell này** — tiles đã sẵn.

> Nếu `data/derived/tiles/data.yaml` đã tồn tại trên Drive, skip cell này.

In [ ]:
import os
tiles_yaml = f'{DATA_ROOT}/derived/tiles/data.yaml'
if os.path.exists(tiles_yaml):
    print('Tiles đã có trên Drive, skip prep:', tiles_yaml)
else:
    !cd /content/pid && PYTHONPATH=src python src/run_pipeline.py prep --step classes data_root=$DATA_ROOT
    !cd /content/pid && PYTHONPATH=src python src/run_pipeline.py prep --step manifest data_root=$DATA_ROOT
    !cd /content/pid && PYTHONPATH=src python src/run_pipeline.py prep --step yolo data_root=$DATA_ROOT detector.label_level=coarse data.label_level=coarse
    !cd /content/pid && PYTHONPATH=src python src/run_pipeline.py prep --step tiles data_root=$DATA_ROOT data.label_level=coarse
    print('Tiles xong.')

In [ ]:
# Kiểm tra tiles
import glob
print('train tiles:', len(glob.glob(f'{DATA_ROOT}/derived/tiles/images/train/*.jpg')))
print('val tiles  :', len(glob.glob(f'{DATA_ROOT}/derived/tiles/images/val/*.jpg')))
!cat $DATA_ROOT/derived/tiles/data.yaml

## 5b. ⚡ Copy tiles Drive → local `/content` (BẮT BUỘC — nếu không sẽ cháy quota)

**Vấn đề đã gặp**: đọc 27k tile trực tiếp từ Drive mounted cực chậm (~0.2 MB/s) — chỉ *scan* dataset đã mất **3.5 tiếng**, cháy hết GPU quota trước khi train.

**Fix**: copy tiles về local disk Colab (`/content`, nhanh gấp trăm lần) **một lần đầu mỗi session**, rồi train đọc từ local. `/content` bị xoá khi session kết thúc nên cell này chạy lại mỗi session — nhưng chỉ ~5-10 phút, rẻ hơn nhiều so với train-từ-Drive.

Cell này tự sinh `data.yaml` local (đường dẫn trỏ `/content`, không phải Drive).

In [ ]:
import os, shutil, glob, time, yaml

# Local data_root: detect_train chỉ cần derived/tiles/ (data.yaml + images + labels).
# Đặt tiles vào đúng cấu trúc data_root local để train với data_root=$LOCAL_DATA_ROOT,
# KHÔNG phải sửa code — detect_train đọc {data_root}/derived/tiles/data.yaml.
LOCAL_DATA_ROOT = '/content/data'
LOCAL_TILES     = f'{LOCAL_DATA_ROOT}/derived/tiles'
DRIVE_TILES     = f'{DATA_ROOT}/derived/tiles'
DRIVE_ZIP       = f'{DATA_ROOT}/derived/tiles.zip'

t0 = time.time()
os.makedirs(f'{LOCAL_DATA_ROOT}/derived', exist_ok=True)

if os.path.exists(DRIVE_ZIP):
    # Nhanh nhất: giải nén 1 file zip (tạo bởi cell 5c) vào local.
    print('Giải nén tiles.zip -> local ...')
    !unzip -q -o "$DRIVE_ZIP" -d "$LOCAL_DATA_ROOT/derived/"
else:
    # Chưa có zip: copy thư mục rời (chậm hơn). Chạy cell 5c 1 lần để tạo zip cho session sau.
    print('Chưa có tiles.zip; copy thư mục rời (chậm). Nên chạy cell 5c 1 lần.')
    if not os.path.isdir(LOCAL_TILES):
        shutil.copytree(DRIVE_TILES, LOCAL_TILES)

# Sinh lại data.yaml LOCAL với đường dẫn trỏ /content (data.yaml gốc trỏ Drive).
with open(f'{DRIVE_TILES}/data.yaml') as f:
    base = yaml.safe_load(f)
local_yaml = {'nc': base['nc'], 'names': base['names']}
for split in ('train', 'val', 'test'):
    d = f'{LOCAL_TILES}/images/{split}'
    if os.path.isdir(d):
        local_yaml[split] = d
with open(f'{LOCAL_TILES}/data.yaml', 'w') as f:
    yaml.safe_dump(local_yaml, f, sort_keys=False)

print(f'Local tiles sẵn sàng sau {time.time()-t0:.0f}s')
print('  train:', len(glob.glob(f'{LOCAL_TILES}/images/train/*.jpg')),
      '| val:', len(glob.glob(f'{LOCAL_TILES}/images/val/*.jpg')))
print('  LOCAL_DATA_ROOT =', LOCAL_DATA_ROOT, '(dùng cho train)')

In [ ]:
# 5c. (TÙY CHỌN, chạy 1 LẦN) Tạo tiles.zip trên Drive để session sau giải nén nhanh.
# Chạy cell này SAU khi đã có tiles local (cell 5b copy rời lần đầu). Lần sau cell 5b
# sẽ thấy zip và giải nén thay vì copy rời -> nhanh hơn nhiều.
import os
if not os.path.exists(DRIVE_ZIP):
    print('Nén tiles -> Drive (1 lần, ~vài phút)...')
    !cd "$LOCAL_DATA_ROOT/derived" && zip -q -r tiles.zip tiles && cp tiles.zip "$DRIVE_ZIP"
    print('Đã tạo', DRIVE_ZIP)
else:
    print('tiles.zip đã có trên Drive:', DRIVE_ZIP)

## 6. Train Stage 1 detector (coarse, 8 nhóm)

**Phiên đầu**: train từ đầu. Đặt `epochs` tổng lớn; nếu Colab ngắt (free ~4-6h/session), phiên sau resume (cell 7). Checkpoint ghi vào `RUNS_DIR` trên Drive nên sống sót.

**VRAM chống tràn**: `detector.batch=-1` → Ultralytics **auto-batch** (tự dò ~60% VRAM của GPU Colab cấp, chọn batch tối đa an toàn). Không phải đoán, không lo `CUDA out of memory` trên T4 free. Nếu vẫn muốn cố định, đổi thành `batch=8` (an toàn) hoặc `batch=16` (nhanh hơn, sát ngưỡng T4).

In [ ]:
# data_root=$LOCAL_DATA_ROOT -> đọc tiles từ /content (nhanh); run_dir=$RUNS_DIR -> checkpoint ra Drive (bền).
!cd /content/pid && PYTHONPATH=src python src/run_pipeline.py detect --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR \
    detector.label_level=coarse detector.epochs=100 detector.batch=-1 detector.imgsz=1024

## 7. Resume train (phiên sau khi Colab ngắt)

Sửa `RESUME_DIR` = run dir phiên trước (in ra ở cuối cell 6, dạng `runs/2026-...`). Resume nạp `last.pt` + optimizer/LR/epoch, train tiếp tới `epochs` gốc — KHÔNG truyền lại epochs/aug.

In [ ]:
# QUAN TRỌNG: chạy cell 5b TRƯỚC (để có tiles local ở /content/data) rồi mới resume —
# YOLO resume đọc lại data path từ run cũ (trỏ /content/data/derived/tiles), phải tồn tại.
RESUME_DIR = f'{RUNS_DIR}/2026-XX-XX_XX-XX-XX'  # <-- sửa thành run dir phiên trước
!cd /content/pid && PYTHONPATH=src python src/run_pipeline.py detect --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR \
    detector.resume=true detector.resume_dir=$RESUME_DIR

## 8. Confusion matrix coarse — QUYẾT ĐỊNH class-agnostic

Ultralytics tự sinh `confusion_matrix.png` + `results.csv` trong run dir khi train xong.
Đây là căn cứ để quyết có nên chuyển Stage 1 sang **class-agnostic** hay không:

- **8 nhóm coarse phân biệt TỐT** (đường chéo đậm, ít nhầm chéo) → detector học coarse ổn → **GIỮ 8 coarse**, class-agnostic không cần.
- **Nhầm chéo NHIỀU** giữa các nhóm (vd valve ↔ safety_valve) → detector chật vật với coarse → cân nhắc **class-agnostic** (1 class 'symbol', dồn phân loại cho Stage 2).

Sửa `RUN_DIR` = run dir vừa train (dạng `.../runs/2026-.../train`).

In [ ]:
from IPython.display import Image as IPImage, display
import os, pandas as pd

RUN_DIR = f'{RUNS_DIR}/2026-XX-XX_XX-XX-XX/train'  # <-- sửa thành run dir vừa train

# Confusion matrix (bản normalized dễ đọc hơn nếu có)
shown = False
for name in ('confusion_matrix_normalized.png', 'confusion_matrix.png'):
    p = os.path.join(RUN_DIR, name)
    if os.path.exists(p):
        print(name); display(IPImage(p)); shown = True; break
if not shown:
    print('Chưa thấy confusion_matrix — train xong chưa? RUN_DIR:', RUN_DIR)

# Per-class mAP từ results.csv (dòng cuối = epoch gần nhất)
csv = os.path.join(RUN_DIR, 'results.csv')
if os.path.exists(csv):
    df = pd.read_csv(csv)
    df.columns = [c.strip() for c in df.columns]
    cols = [c for c in df.columns if 'mAP' in c or 'precision' in c.lower() or 'recall' in c.lower()]
    print('\nMetrics (epoch cuối):')
    print(df[cols].iloc[-1].to_string())

## 9. (Sau khi train xong) Inference + eval

Sửa `BEST` = đường dẫn `best.pt` (in ra cuối cell 6/7).

In [ ]:
BEST = f'{RUNS_DIR}/2026-XX-XX_XX-XX-XX/train/weights/best.pt'  # <-- sửa thành run dir thật

# QUAN TRỌNG:
#  - eval.split=val cho CẢ infer LẪN evaluate — DigitizePID chỉ có train/val, KHÔNG có test
#    (config mặc định eval.split=test -> infer ra 0 ảnh nếu không override).
#  - data_root=$DATA_ROOT (Drive) — infer SAHI đọc ẢNH GỐC canonical/images (7168px), KHÔNG
#    dùng tiles. $LOCAL_DATA_ROOT chỉ có tiles nên sẽ không tìm thấy ảnh.
#  - Lưu ý: đọc 100 ảnh val 7168px từ Drive hơi chậm (I/O). Nếu quá chậm, báo mình để thêm
#    cell copy canonical val về local.
!cd /content/pid && PYTHONPATH=src python src/run_pipeline.py detect --infer \
    data_root=$DATA_ROOT run_dir=$RUNS_DIR \
    detector.ckpt=$BEST detector.label_level=coarse eval.split=val

!cd /content/pid && PYTHONPATH=src python src/run_pipeline.py evaluate \
    data_root=$DATA_ROOT run_dir=$RUNS_DIR eval.mode=det_only eval.split=val

## 10. Visualize kết quả detect (vẽ box lên ảnh)

Vẽ box detector tìm được lên ảnh P&ID gốc — xem bằng mắt + lấy hình cho luận văn. Cần
chạy **cell 9 (infer)** trước để có detection `.txt`.

- `--mode pred` — chỉ box dự đoán (mặc định).
- `--mode compare` — overlay **GT (xanh) vs prediction (đỏ)** để so sánh đúng/sai.
- Xuất **full-ảnh** (downscale) + **crop zoom** vài vùng (symbol 7168px nhìn mới rõ) vào `derived/viz/val/`.

`data_root=$DATA_ROOT` (Drive) vì cần ảnh canonical gốc.

In [ ]:
# Vẽ box theo MÀU class + legend panel, LỌC score>=0.25 (bỏ box rác score thấp).
#   --mode pred     : box dự đoán, màu theo class (legend bên phải)
#   --mode compare  : GT (xanh) vs pred (đỏ) — xem detector đúng/sai
#   --fill          : thêm highlight nhạt trong box (mặc định chỉ border)
#   --score-thr 0.25: chỉ vẽ box đáng tin (khớp eval). Hạ xuống để xem cả box score thấp.
!cd /content/pid && PYTHONPATH=src python scripts/visualize_detections.py \
    --data-root $DATA_ROOT --split val --mode pred --max-images 5 --score-thr 0.25

# Hiển thị vài ảnh full (có legend) + crop ngay trong notebook
import glob
from IPython.display import Image as IPImage, display
viz_dir = f'{DATA_ROOT}/derived/viz/val'
fulls = sorted(glob.glob(f'{viz_dir}/*_full.jpg'))[:2]
crops = sorted(glob.glob(f'{viz_dir}/crops/*.jpg'))[:4]
for p in fulls + crops:
    print(p.split('/')[-1]); display(IPImage(p))